# Cost-Efficient VLM Rubric Grader — Proof of Concept

Pre-application proof-of-concept for the [TAP Cost-Efficient AI Model project](https://github.com/theapprenticeproject/C4GT_2026/issues/2) in **DMP 2026**.

This notebook runs **zero-shot Qwen2-VL-2B-Instruct** against a sample student artifact, parses its structured rubric output, and reports the JSON-validity rate. It establishes the baseline number that the proposal's fine-tuning effort improves on.

**Author:** Neha Dhruw ([@neha222222](https://github.com/neha222222))

## What this notebook proves

1. A small open-source VLM (2B params, ~3 GB VRAM in 4-bit) can produce rubric-shaped JSON without fine-tuning.
2. The rubric prompt + structured-output schema parse end-to-end on a Colab T4 GPU.
3. A QWK metric is wired in for downstream eval against human / Gemini scores.

Bottlenecks the proposal's fine-tuning addresses (visible in this notebook's outputs):
- Format-validity rate is **<100 % zero-shot** — fine-tuning targets ≥99 %.
- Per-dimension scores are noisy without rubric-aligned supervision.
- Cost per assessment is GPU-dominated — this notebook measures the baseline.

## 0. Setup (Colab)

Requires a GPU runtime. On Colab: `Runtime → Change runtime type → T4 GPU`. Approx 4 minutes for the model download on first run.

In [ ]:
# Colab: clone repo so `rubric.py` is importable.
import os
if 'COLAB_GPU' in os.environ or not os.path.exists('rubric.py'):
    !git clone -q https://github.com/neha222222/tap-vlm-demo.git
    %cd tap-vlm-demo

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes pillow

## 1. Load Qwen2-VL-2B-Instruct in 4-bit

BitsAndBytes NF4 quantization keeps VRAM under ~3 GB on a T4. The same setup is the starting point for the proposed LoRA fine-tune.

In [ ]:
import torch
from transformers import (
    AutoProcessor,
    Qwen2VLForConditionalGeneration,
    BitsAndBytesConfig,
)

MODEL_ID = 'Qwen/Qwen2-VL-2B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
model.eval()
print(f'Model loaded. VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB')

## 2. Build the rubric prompt + load a sample artifact

We use a publicly-licensed sample image of a student drawing as a stand-in for TAP's actual dataset. The full pipeline is identical.

In [ ]:
import io
import requests
from PIL import Image

from rubric import build_prompt, parse_vlm_output

# Public-domain sample (Wikimedia Commons child drawing).
SAMPLE_IMAGE_URLS = [
    'https://upload.wikimedia.org/wikipedia/commons/thumb/9/95/Mexican_kid_drawing.jpg/640px-Mexican_kid_drawing.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Children_drawings_kids.jpg/640px-Children_drawings_kids.jpg',
]

def load_image(url):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return Image.open(io.BytesIO(r.content)).convert('RGB')

prompt = build_prompt()
print(prompt)

## 3. Inference helper

Single-image inference. Batched inference is part of the proposed cost-engineering pass.

In [ ]:
import time

def grade_artifact(image, prompt_text):
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image'},
                {'type': 'text', 'text': prompt_text},
            ],
        }
    ]
    chat = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=[chat], images=[image], return_tensors='pt').to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    elapsed = time.time() - t0
    raw = processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0]
    return raw, elapsed

## 4. Run the rubric grader on sample artifacts

Two samples is enough to demonstrate the pipeline + measure baseline JSON-validity. The proposal scales this to gold (200) + silver (2000) + adversarial (100) eval sets.

In [ ]:
results = []
for url in SAMPLE_IMAGE_URLS:
    print(f'\n--- {url}')
    img = load_image(url)
    raw, elapsed = grade_artifact(img, prompt)
    print(f'Raw output ({elapsed:.2f}s):')
    print(raw)
    try:
        score = parse_vlm_output(raw)
        print('Parsed:', score.to_dict())
        results.append({'url': url, 'parsed': True, 'score': score, 'latency_s': elapsed})
    except ValueError as exc:
        print(f'Parse failed: {exc}')
        results.append({'url': url, 'parsed': False, 'score': None, 'latency_s': elapsed})

valid = sum(1 for r in results if r['parsed'])
print(f'\nFormat-validity rate: {valid}/{len(results)} = {valid/len(results):.0%}')
print(f'Mean latency: {sum(r["latency_s"] for r in results)/len(results):.2f} s')

## 5. Cost projection

Working from the latency measured above, project per-assessment cost on a serverless T4 GPU at the typical Indian cloud price of ₹50/hour.

In [ ]:
GPU_HOURLY_INR = 50.0
mean_latency = sum(r['latency_s'] for r in results) / len(results)
cost_per_assessment_inr = (mean_latency / 3600) * GPU_HOURLY_INR

print(f'Mean latency:          {mean_latency:.2f} s')
print(f'Cost / assessment:     ₹{cost_per_assessment_inr:.4f}')
print('Target (issue):        ₹0.10')
print(f'Headroom:              {0.10 / cost_per_assessment_inr:.1f}x' if cost_per_assessment_inr > 0 else '')
print('\nThe proposal\'s cost-engineering pass (batched inference, image resize,')
print('autoscale-to-zero, response cache) is designed to drop this ~10x further.')

## 6. Inter-rater agreement on a synthetic eval

QWK on a small synthetic example, demonstrating the metric the full eval pipeline uses.

In [ ]:
from rubric import quadratic_weighted_kappa

# 8 artifacts, two raters mostly agreeing.
human_scores  = [1, 2, 3, 4, 1, 2, 3, 4]
model_scores  = [1, 2, 3, 4, 2, 2, 3, 4]
qwk = quadratic_weighted_kappa(human_scores, model_scores)
print(f'QWK = {qwk:.3f}  (proposal acceptance bar: >= 0.65 vs human gold)')

## What this notebook does NOT do

Honest scoping for the mentor:
- No fine-tuning yet — that's the body of the DMP work.
- No batched inference — single-sample only, so cost is an upper bound.
- No human gold labels for these images — QWK demo uses synthetic ratings.
- No constrained decoding (`outlines` / `xgrammar`) — relies on prompt-only structured output, which is exactly why format-validity is <100 % zero-shot.

Each item maps to a milestone in the proposal's 12-week timeline.